# 10 — Average Order Value (AOV) by Segment
Average revenue and profit per order for each customer segment, with margin trend.


In [ ]:
import os, sys
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns

os.environ['JAVA_HOME']             = '/usr/local/java'
os.environ['SPARK_HOME']            = '/usr/local/spark'
os.environ['HADOOP_CONF_DIR']       = '/usr/local/hadoop/etc/hadoop'
os.environ['PYSPARK_PYTHON']        = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

try:
    spark.stop()
except:
    pass

# Local mode — no Hive/hive-metastore dependency
spark = (SparkSession.builder
    .appName("Superstore Analytics")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# ── Load CSVs and register temp views ─────────────────────────────────────────
DATA = "/usr/local/hadoop/etc/hadoop/assessment-2"

def load(filename, renames):
    df = spark.read.option("header", "true").option("inferSchema", "true") \
             .csv(f"{DATA}/{filename}")
    for old, new in renames.items():
        df = df.withColumnRenamed(old, new)
    return df

customers  = load("customers.csv",  {"Customer ID": "customer_id",
                                      "Customer Name": "customer_name",
                                      "Segment": "segment"})
orders     = load("orders.csv",     {"Order ID": "order_id",
                                      "Order Date": "order_date",
                                      "Ship Date": "ship_date",
                                      "Ship Mode": "ship_mode",
                                      "Customer ID": "customer_id",
                                      "Postal Code": "postal_code"})
order_items = load("order_items.csv", {"Row ID": "row_id",
                                        "Order ID": "order_id",
                                        "Product ID": "product_id",
                                        "Sales": "sales",
                                        "Quantity": "quantity",
                                        "Discount": "discount",
                                        "Profit": "profit"})
products   = load("products.csv",   {"Product ID": "product_id",
                                      "Product Name": "product_name",
                                      "Category": "category",
                                      "Sub-Category": "sub_category"})
locations  = load("locations.csv",  {"Postal Code": "postal_code",
                                      "City": "city",
                                      "State": "state",
                                      "Country": "country",
                                      "Region": "region"})

customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
order_items.createOrReplaceTempView("order_items")
products.createOrReplaceTempView("products")
locations.createOrReplaceTempView("locations")

print("Spark", spark.version, "ready — all tables loaded.")
spark.sql("SHOW TABLES").show()

# ── Global chart style ─────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
PALETTE = sns.color_palette("muted")

def fmt_usd(v):
    if abs(v) >= 1_000_000:
        return f"${v/1_000_000:.2f}M"
    if abs(v) >= 1_000:
        return f"${v/1_000:.1f}K"
    return f"${v:.0f}"


In [ ]:
# ── Average Order Value (AOV) by Segment ─────────────────────────────────────
aov = spark.sql("""
    SELECT c.segment,
           ROUND(SUM(oi.sales)  / COUNT(DISTINCT o.order_id), 2) AS avg_order_value,
           ROUND(SUM(oi.profit) / COUNT(DISTINCT o.order_id), 2) AS avg_order_profit,
           ROUND(SUM(oi.profit)/SUM(oi.sales)*100, 2)            AS margin_pct,
           COUNT(DISTINCT o.order_id)                             AS total_orders
    FROM customers c
    JOIN orders      o  ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id    = oi.order_id
    GROUP BY c.segment
    ORDER BY avg_order_value DESC
""").toPandas()

bar_w = 0.35
x     = range(len(aov))
fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()
b1 = ax1.bar([i - bar_w/2 for i in x], aov["avg_order_value"],
             bar_w, color=PALETTE[0], label="Avg Order Value",  edgecolor="white", zorder=3)
b2 = ax1.bar([i + bar_w/2 for i in x], aov["avg_order_profit"],
             bar_w, color=PALETTE[1], label="Avg Order Profit", edgecolor="white", zorder=3)
ax2.plot(x, aov["margin_pct"], color=PALETTE[2], marker="D",
         linewidth=2, markersize=8, label="Margin %", zorder=4)
for bar in b1:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             fmt_usd(bar.get_height()),
             ha="center", va="bottom", fontsize=9,
             fontweight="bold", color=PALETTE[0])
for bar in b2:
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             fmt_usd(bar.get_height()),
             ha="center", va="bottom", fontsize=9,
             fontweight="bold", color=PALETTE[1])
for xi, mg in zip(x, aov["margin_pct"]):
    ax2.text(xi, mg + 0.5, f"{mg:.1f}%",
             ha="center", va="bottom", fontsize=9,
             color=PALETTE[2], fontweight="bold")
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(handles1 + handles2, labels1 + labels2,
           loc="upper right", frameon=False, fontsize=9)
ax1.set_xticks(list(x))
ax1.set_xticklabels(aov["segment"], fontsize=11)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: fmt_usd(v)))
ax1.set_title("Average Order Value (AOV) by Customer Segment")
ax1.set_ylabel("USD per Order")
ax2.set_ylabel("Profit Margin %")
ax1.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
plt.tight_layout()
plt.show()


In [ ]:
spark.stop()
print("Spark stopped.")
